In [1]:
import time

time.sleep(60*60)

In [2]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import pickle
from pathlib import Path
from collections import Counter
from itertools import product
from tqdm.auto import tqdm
from sklearn.metrics import mean_squared_error

from statsmodels.tsa.statespace.sarimax import SARIMAX

In [3]:
# ==============================================================================
# Metric Helpers
# ==============================================================================
def smape(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    denom = (np.abs(y_true) + np.abs(y_pred)) / 2.0
    return np.mean(np.abs(y_true - y_pred) / np.maximum(denom, 1e-8))

def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

def mape(y_true, y_pred):
    y_true, y_pred = np.array(y_true, dtype=float), np.array(y_pred, dtype=float)
    mask = y_true != 0
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100

In [4]:
# ==============================================================================
# Paths & Config
# ==============================================================================
TRAIN_PATH = "../../../data/silver_money_calc/train.parquet"
VAL_PATH   = "../../../data/silver_money_calc/val.parquet"
TEST_PATH  = "../../../data/silver_money_calc/test.parquet"

Y_COL     = "Sum of кВт"
GROUP_COL = "EIC-код_cat"

SEASONAL_PERIOD  = 24    # 24-hour daily seasonality
N_OPT_STATIONS   = 10   # stations used for auto_arima parameter search
TARGET_STATIONS  = 10 # 410  # same as TFT for fair comparison

CHECKPOINT_DIR = Path("F:/checkpoints/sarimax")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

In [5]:
# ==============================================================================
# Exogenous Features
# ==============================================================================
# Weather covariates known at forecast time (trimmed to low-correlation subset)
EXOG_WEATHER = [
    "temperature_2m",
    "relative_humidity_2m",
    "precipitation",
    "cloud_cover",
    "wind_speed_10m",
    "shortwave_radiation",
]

# Cyclic calendar encodings (always known in advance)
CALENDAR_COLS = [
    "hour_sin", "hour_cos",
    "dow_sin",  "dow_cos",
    "month_sin", "month_cos",
]

ALL_EXOG = EXOG_WEATHER + CALENDAR_COLS

In [6]:
# ==============================================================================
# Data Loading
# ==============================================================================
def add_calendar_features(df: pd.DataFrame) -> pd.DataFrame:
    # datetime is already tz-naive at this point
    dt    = df["datetime"]
    hour  = dt.dt.hour
    dow   = dt.dt.dayofweek
    month = dt.dt.month
    df["hour_sin"]  = np.sin(2 * np.pi * hour  / 24)
    df["hour_cos"]  = np.cos(2 * np.pi * hour  / 24)
    df["dow_sin"]   = np.sin(2 * np.pi * dow   / 7)
    df["dow_cos"]   = np.cos(2 * np.pi * dow   / 7)
    df["month_sin"] = np.sin(2 * np.pi * month / 12)
    df["month_cos"] = np.cos(2 * np.pi * month / 12)
    return df

def load_data(path: str) -> pd.DataFrame:
    df = pd.read_parquet(path)
    df["datetime"] = pd.to_datetime(df["datetime"], utc=True).dt.tz_convert(None)
    df = df.sort_values([GROUP_COL, "datetime"]).reset_index(drop=True)
    df = add_calendar_features(df)
    return df

print("Loading train …")
train = load_data(TRAIN_PATH)
print(f"  range : {train['datetime'].min()} → {train['datetime'].max()}")
print(f"  rows  : {len(train):,}")

print("Loading val …")
val = load_data(VAL_PATH)
print(f"  range : {val['datetime'].min()} → {val['datetime'].max()}")

print("Loading test …")
test = load_data(TEST_PATH)
print(f"  range : {test['datetime'].min()} → {test['datetime'].max()}")

Loading train …
  range : 2023-12-31 23:00:00 → 2025-06-30 20:00:00
  rows  : 4,864,324
Loading val …
  range : 2025-06-29 22:00:00 → 2025-07-31 20:00:00
Loading test …
  range : 2025-07-30 22:00:00 → 2025-08-31 21:00:00


In [7]:
# ==============================================================================
# Station Sampling  (same seed + count as TFT for fair comparison)
# ==============================================================================
station_stats = (
    train.groupby(GROUP_COL)
    .agg(
        rows        = (Y_COL, "count"),
        target_mean = (Y_COL, "mean"),
        target_std  = (Y_COL, "std"),
    )
    .reset_index()
    .sort_values("rows", ascending=False)
)

print(f"Total stations available: {len(station_stats)}")

np.random.seed(42)
sampled_stations = station_stats.sample(
    n=TARGET_STATIONS, random_state=42
)[GROUP_COL].values

train = train[train[GROUP_COL].isin(sampled_stations)].reset_index(drop=True)
val   = val[val[GROUP_COL].isin(sampled_stations)].reset_index(drop=True)
test  = test[test[GROUP_COL].isin(sampled_stations)].reset_index(drop=True)

print(f"Stations kept : {len(sampled_stations)}")
print(f"Train rows    : {len(train):,}")
print(f"Val rows      : {len(val):,}")
print(f"Test rows     : {len(test):,}")

Total stations available: 410
Stations kept : 10
Train rows    : 122,467
Val rows      : 7,670
Test rows     : 6,936


In [8]:
# ==============================================================================
# Parameter Optimisation — AIC grid search via statsmodels
# Runs on N_OPT_STATIONS stations using the most recent OPT_SLICE_HOURS hours.
# Searches over (p,d,q) x (P,D,Q) and picks the most-common best orders.
# ==============================================================================
import warnings

OPT_SLICE_HOURS = 24 * 30   # 30 days — enough signal, fast to fit

P_RANGE  = range(0, 2)
D_RANGE  = range(0, 2)
Q_RANGE  = range(0, 2)
SP_RANGE = range(0, 2)
SD_RANGE = range(0, 2)
SQ_RANGE = range(0, 2)

CANDIDATE_ORDERS   = list(product(P_RANGE,  D_RANGE,  Q_RANGE))
CANDIDATE_SEASONAL = list(product(SP_RANGE, SD_RANGE, SQ_RANGE))

opt_stations = sampled_stations[:N_OPT_STATIONS]
opt_results  = []

for eic in tqdm(opt_stations, desc="Grid search"):
    s = train[train[GROUP_COL] == eic].sort_values("datetime")
    y = s[Y_COL].values[-OPT_SLICE_HOURS:]
    X = s[ALL_EXOG].values[-OPT_SLICE_HOURS:]

    best_aic    = np.inf
    best_order  = None
    best_sorder = None

    for order in CANDIDATE_ORDERS:
        for sorder in CANDIDATE_SEASONAL:
            try:
                with warnings.catch_warnings():
                    warnings.simplefilter("ignore")
                    res = SARIMAX(
                        y,
                        exog                  = X,
                        order                 = order,
                        seasonal_order        = sorder + (SEASONAL_PERIOD,),
                        enforce_stationarity  = False,
                        enforce_invertibility = False,
                    ).fit(disp=False, maxiter=200)
                if res.aic < best_aic:
                    best_aic    = res.aic
                    best_order  = order
                    best_sorder = sorder + (SEASONAL_PERIOD,)
            except Exception:
                continue

    if best_order is not None:
        opt_results.append({
            "eic"           : eic,
            "order"         : best_order,
            "seasonal_order": best_sorder,
            "aic"           : best_aic,
        })
        print(f"{eic}: order={best_order}  seasonal={best_sorder}  AIC={best_aic:.1f}")
    else:
        print(f"{eic}: all candidates failed")

opt_df = pd.DataFrame(opt_results)
print("\n", opt_df)

Grid search:   0%|          | 0/10 [00:00<?, ?it/s]

62Z7973386804189: order=(1, 0, 1)  seasonal=(0, 1, 1, 24)  AIC=-68.5
62Z8410886810640: order=(1, 1, 1)  seasonal=(1, 1, 1, 24)  AIC=829.2
62Z9672876874134: order=(1, 0, 1)  seasonal=(1, 1, 1, 24)  AIC=88.8
62Z9214767819161: order=(1, 0, 1)  seasonal=(1, 1, 1, 24)  AIC=513.6
62Z5544769493288: order=(1, 0, 1)  seasonal=(0, 1, 1, 24)  AIC=847.5
62Z6960657066877: order=(1, 0, 1)  seasonal=(1, 1, 1, 24)  AIC=-105.5
62Z8426544240206: order=(1, 0, 1)  seasonal=(0, 1, 1, 24)  AIC=1530.9
62Z4108523399069: order=(1, 0, 1)  seasonal=(0, 1, 1, 24)  AIC=-41.5
62Z3777584435448: order=(1, 0, 1)  seasonal=(0, 1, 1, 24)  AIC=-290.5
62Z220653865958K: order=(1, 0, 1)  seasonal=(0, 1, 1, 24)  AIC=843.8

                 eic      order seasonal_order          aic
0  62Z7973386804189  (1, 0, 1)  (0, 1, 1, 24)   -68.530820
1  62Z8410886810640  (1, 1, 1)  (1, 1, 1, 24)   829.231031
2  62Z9672876874134  (1, 0, 1)  (1, 1, 1, 24)    88.773623
3  62Z9214767819161  (1, 0, 1)  (1, 1, 1, 24)   513.556304
4  62Z55447

In [9]:
# ==============================================================================
# Select Best Common Orders
# ==============================================================================
order_counts    = Counter(opt_df["order"].tolist())
seasonal_counts = Counter(opt_df["seasonal_order"].tolist())

BEST_ORDER          = order_counts.most_common(1)[0][0]
BEST_SEASONAL_ORDER = seasonal_counts.most_common(1)[0][0]

print(f"Most common order          : {BEST_ORDER}")
print(f"Most common seasonal order : {BEST_SEASONAL_ORDER}")
print()
print("Order distribution:")
for o, cnt in order_counts.most_common():
    print(f"  {o}  →  {cnt} stations")
print()
print("Seasonal order distribution:")
for o, cnt in seasonal_counts.most_common():
    print(f"  {o}  →  {cnt} stations")

Most common order          : (1, 0, 1)
Most common seasonal order : (0, 1, 1, 24)

Order distribution:
  (1, 0, 1)  →  9 stations
  (1, 1, 1)  →  1 stations

Seasonal order distribution:
  (0, 1, 1, 24)  →  6 stations
  (1, 1, 1, 24)  →  4 stations


In [10]:
# ==============================================================================
# Fit SARIMAX on All Stations
# Checkpoints each fitted model so the cell is resumable.
# ==============================================================================
models  = {}
failed  = []

for eic in tqdm(sampled_stations, desc="Fitting SARIMAX"):
    ckpt = CHECKPOINT_DIR / f"{eic}.pkl"

    if ckpt.exists():
        try:
            with open(ckpt, "rb") as f:
                models[eic] = pickle.load(f)
            continue
        except Exception as e:
            print(f"  {eic}: corrupted checkpoint, retraining – {e}")
            ckpt.unlink()

    s = train[train[GROUP_COL] == eic].sort_values("datetime")
    y = s[Y_COL].values
    X = s[ALL_EXOG].values

    try:
        res = SARIMAX(
            y,
            exog                 = X,
            order                = BEST_ORDER,
            seasonal_order       = BEST_SEASONAL_ORDER,
            enforce_stationarity = False,
            enforce_invertibility= False,
        ).fit(disp=False)

        with open(ckpt, "wb") as f:
            pickle.dump(res, f)

        models[eic] = res
    except Exception as e:
        print(f"  {eic}: fit failed – {e}")
        failed.append(eic)

print(f"\nFitted  : {len(models)}")
print(f"Failed  : {len(failed)}")

Fitting SARIMAX:   0%|          | 0/10 [00:00<?, ?it/s]

C:\Users\Lev\Miniconda3\envs\Diploma\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


  62Z7973386804189: fit failed – 


C:\Users\Lev\Miniconda3\envs\Diploma\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


  62Z8410886810640: fit failed – 


...

  62Z4108523399069: fit failed – Unable to allocate 250. MiB for an array with shape (50, 50, 13126) and data type float64


C:\Users\Lev\Miniconda3\envs\Diploma\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


  62Z3777584435448: fit failed – Unable to allocate 250. MiB for an array with shape (50, 50, 13126) and data type float64


C:\Users\Lev\Miniconda3\envs\Diploma\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


  62Z220653865958K: fit failed – Unable to allocate 250. MiB for an array with shape (50, 50, 13126) and data type float64

Fitted  : 1
Failed  : 9


In [11]:
# ==============================================================================
# Predict – Validation Set
# model.forecast() extends prediction beyond the training window.
# ==============================================================================
val_preds = []

for eic in tqdm(sampled_stations, desc="Val predictions"):
    if eic not in models:
        continue
    v = val[val[GROUP_COL] == eic].sort_values("datetime")
    if v.empty:
        continue

    X_val    = v[ALL_EXOG].values
    forecast = models[eic].forecast(steps=len(v), exog=X_val)
    forecast = np.maximum(forecast, 0)   # clip negative predictions

    val_preds.append(pd.DataFrame({
        GROUP_COL : eic,
        "datetime": v["datetime"].values,
        "pred"    : forecast,
        Y_COL     : v[Y_COL].values,
    }))

val_eval = pd.concat(val_preds, ignore_index=True)
print(f"Aligned samples: {len(val_eval):,}")

Val predictions:   0%|          | 0/10 [00:00<?, ?it/s]

Aligned samples: 767


In [12]:
# ==============================================================================
# Validation Metrics
# ==============================================================================
print("── Validation ──────────────────────────────────────────────")
print(f"Aligned samples : {len(val_eval):,}")
print(f"SMAPE : {smape(val_eval[Y_COL], val_eval['pred']):.4f}")
print(f"RMSE  : {rmse(val_eval[Y_COL],  val_eval['pred']):.4f}")
print(f"MAPE  : {mape(val_eval[Y_COL],  val_eval['pred']):.2f} %")

── Validation ──────────────────────────────────────────────
Aligned samples : 767
SMAPE : 0.4951
RMSE  : 11.7350
MAPE  : 67.89 %


In [13]:
# ==============================================================================
# Predict – Test Set
# Update each model's state through the validation period first (no refit),
# then forecast across the test horizon.
# ==============================================================================
test_preds = []

for eic in tqdm(sampled_stations, desc="Test predictions"):
    if eic not in models:
        continue

    v = val[val[GROUP_COL]   == eic].sort_values("datetime")
    t = test[test[GROUP_COL] == eic].sort_values("datetime")
    if t.empty:
        continue

    X_test = t[ALL_EXOG].values

    try:
        # Advance the Kalman filter state through val without refitting
        if not v.empty:
            X_val          = v[ALL_EXOG].values
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                updated_result = models[eic].apply(v[Y_COL].values, exog=X_val, refit=False)
        else:
            updated_result = models[eic]

        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            forecast = updated_result.forecast(steps=len(t), exog=X_test)
        forecast = np.maximum(forecast, 0)

        test_preds.append(pd.DataFrame({
            GROUP_COL : eic,
            "datetime": t["datetime"].values,
            "pred"    : forecast,
            Y_COL     : t[Y_COL].values,
        }))
    except Exception as e:
        print(f"  {eic}: forecast failed — {e}")

if test_preds:
    test_eval = pd.concat(test_preds, ignore_index=True)
    print(f"Aligned samples: {len(test_eval):,}")
else:
    print("No test predictions — check that models were fitted in this session.")

Test predictions:   0%|          | 0/10 [00:00<?, ?it/s]

Aligned samples: 768


In [14]:
# ==============================================================================
# Test Metrics
# ==============================================================================
print("── Test ────────────────────────────────────────────────────")
print(f"Aligned samples : {len(test_eval):,}")
print(f"SMAPE : {smape(test_eval[Y_COL], test_eval['pred']):.4f}")
print(f"RMSE  : {rmse(test_eval[Y_COL],  test_eval['pred']):.4f}")
print(f"MAPE  : {mape(test_eval[Y_COL],  test_eval['pred']):.2f} %")

── Test ────────────────────────────────────────────────────
Aligned samples : 768
SMAPE : 0.4008
RMSE  : 8.9106
MAPE  : 51.53 %


In [15]:
# ==============================================================================
# Per-Station Metrics Summary
# ==============================================================================
def per_station_metrics(eval_df):
    rows = []
    for eic, g in eval_df.groupby(GROUP_COL):
        rows.append({
            GROUP_COL  : eic,
            "n"        : len(g),
            "SMAPE"    : smape(g[Y_COL], g["pred"]),
            "RMSE"     : rmse(g[Y_COL],  g["pred"]),
            "MAPE"     : mape(g[Y_COL],  g["pred"]),
        })
    return pd.DataFrame(rows).sort_values("RMSE", ascending=False)

val_station_metrics  = per_station_metrics(val_eval)
test_station_metrics = per_station_metrics(test_eval)

print("── Worst 10 stations (test RMSE) ───────────────────────────")
print(test_station_metrics.head(10).to_string(index=False))

── Worst 10 stations (test RMSE) ───────────────────────────
     EIC-код_cat   n    SMAPE     RMSE      MAPE
62Z8426544240206 768 0.400755 8.910572 51.533901


In [16]:
# ==============================================================================
# Forecast Plot
# ==============================================================================
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

def plot_forecast(eval_df, eic_code, start_dt=None, end_dt=None):
    df = eval_df[eval_df[GROUP_COL] == eic_code].sort_values("datetime")
    if df.empty:
        raise ValueError(f"No data for {eic_code!r}")
    if start_dt:
        df = df[df["datetime"] >= pd.Timestamp(start_dt)]
    if end_dt:
        df = df[df["datetime"] <= pd.Timestamp(end_dt)]
    if df.empty:
        raise ValueError("No data in the specified datetime range.")

    fig, ax = plt.subplots(figsize=(14, 4))
    ax.plot(df["datetime"], df[Y_COL],  label="True",      linewidth=1, color="steelblue")
    ax.plot(df["datetime"], df["pred"], label="Predicted", linewidth=1, color="tomato", alpha=0.85)
    ax.set_title(
        f"{eic_code}  |  MAPE={mape(df[Y_COL], df['pred']):.2f}%"
        f"  RMSE={rmse(df[Y_COL], df['pred']):.2f}"
        f"  ({df['datetime'].min().date()} – {df['datetime'].max().date()})"
    )
    ax.set_xlabel("Datetime")
    ax.set_ylabel(Y_COL)
    ax.legend()
    ax.xaxis.set_major_locator(mdates.AutoDateLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%m-%d %H:%M"))
    fig.autofmt_xdate(rotation=0, ha="center")
    plt.tight_layout()
    plt.show()

# Usage:
# plot_forecast(val_eval,  eic_code=sampled_stations[0])
# plot_forecast(test_eval, eic_code=sampled_stations[0], start_dt="2025-08-01", end_dt="2025-08-07")
plot_forecast(test_eval, eic_code=sampled_stations[0])

ValueError: No data for '62Z7973386804189'